# Extract GFM Inundation from STAC

This notebook extract GFM data using STAC, load it into a xarray and calculate the maximum flood extent over a time range for an area is calculated.

## First the imports

In [ ]:
import pyproj
import rioxarray # noqa
import xarray as xr
from datetime import datetime, timedelta
from shapely.geometry import box
from pystac_client import Client
from odc import stac as odc_stac

## Search and load data

We will define our area (AOI) and time range of interest for which we want
to calculate the maximum flood extent for. For defining a bounding box, you can
use [this web tool](http://bboxfinder.com).

All GFM data is registered as a [STAC](https://stacspec.org/en/) collection.
Please find more information about STAC in our [documentation](https://docs.eodc.eu/services/stac.html).

In [ ]:
def search_subdomain(area,date,days):
# Define the API URL to search the GFM stac
    api_url = "https://stac.eodc.eu/api/v1"

# Define the STAC collection ID
    collection_id = "GFM"

    aoi = box(area[1], area[0], area[3], area[2])
    s = str(date)
    year  = int(s[0:4])
    month = int(s[4:6])
    day   = int(s[6:8])
# Define the time range for the search
#  time_range = (datetime(year, month, day, 0), datetime(year, month, day+days-1, 23))
    start = datetime(year, month, day, 0, 0, 0)
    end   = start + timedelta(days=days) - timedelta(seconds=1)  # inclusive end
    print ("Extraction period from",start,"to",end)
    time_range = (start, end)
# Open the STAC catalog using the specified API URL
    eodc_catalog = Client.open(api_url)

# Perform a search in the catalog with the specified parameters
    search = eodc_catalog.search(
        max_items=1000,             # Maximum number of items to return
        collections=collection_id,  # The collection to search within
        intersects=aoi,             # The area of interest
        datetime=time_range         # The time range for the search
    )

# Collect the found items into an item collection
    items = search.item_collection()

    return items

The found STAC items loaded into a xarray.Dataset object to calculate the maximum flood extent. The "ensemble_flood_extent" layer of each GFM item. Furthermore, one needs to specify
the coordinate reference system (CRS) as well as the resolution of the data. All
necessary metadata is saved in each STAC item.

In [ ]:
def extract_subdomain(aa, items, obs, msk):
    import odc.stac
    import rioxarray as rxr
    import numpy as np
    import xarray as xr
    from shapely.geometry import box
    import pyproj

    crs_src = pyproj.CRS.from_wkt(items[0].properties["proj:wkt2"])
    resolution = items[0].properties["gsd"]
    aoi = box(aa[1], aa[0], aa[3], aa[2])

    factor = 4
    sum_flood = None
    sum_missing = None

    print("\n=== Running flood extraction ===")

    for idx, titem in enumerate(items):

        xx = odc.stac.load(
            [titem],
            bbox=aoi.bounds,
            crs=crs_src,
            bands=["ensemble_flood_extent", "reference_water_mask"],
            resolution=resolution,
            dtype="uint8",
            groupby="solar_day",
            chunks={}
        )

        flood_ds = xx["ensemble_flood_extent"].coarsen(
            y=factor, x=factor, boundary="trim"
        ).max()

        perm_ds = xx["reference_water_mask"].coarsen(
            y=factor, x=factor, boundary="trim"
        ).max()

        ds = xr.Dataset({
            "flood": flood_ds,
            "perm": perm_ds
        }).rio.write_crs(crs_src)

        ds_ll = ds.rio.reproject(
            "EPSG:4326",
            nodata=255,
            resampling="average"
        )
        del xx, flood_ds, perm_ds, ds

        flood_arr = ds_ll["flood"].fillna(255).astype("uint16").values
        perm_arr  = ds_ll["perm"].fillna(255).astype("uint16").values

        if sum_flood is None:
            shape = flood_arr.shape
            sum_flood = np.zeros(shape, dtype="float32")
            sum_missing = np.zeros(shape, dtype="uint16")
            coords = ds_ll["flood"].coords
            dims = ds_ll["flood"].dims

        flood = np.where((flood_arr != 255) & (flood_arr != 0), flood_arr, 0).astype("float32")

        missing = np.where(
            ((flood_arr != 255) | (perm_arr != 255)) &
            ((flood_arr != 0) | (perm_arr != 0)),
            1, 0
        ).astype("uint16")

        sum_flood += flood
        sum_missing += missing

        del ds_ll, flood_arr, perm_arr, flood, missing
        print(f"Item {idx+1}/{len(items)} OK")

    obs_da = xr.DataArray(
        sum_flood,
        coords=coords,
        dims=dims
    ).rio.write_crs("EPSG:4326")

    obs_da = obs_da.fillna(0)
    obs_da.values[np.isnan(obs_da.values)] = 0
    obs_da.rio.to_raster(obs, compress="LZW")

    msk_da = xr.DataArray(
        (sum_missing >= 1).astype("uint8"),
        coords=coords,
        dims=dims
    ).rio.write_crs("EPSG:4326")

    msk_da.values[np.isnan(msk_da.values)] = 0
    msk_da.rio.to_raster(msk, compress="LZW")

    print("\n=== DONE ===\n")
    return crs_src

In [ ]:
# Extract GFM on AREA and write Obs and Mask files def extract_subdomain(aa, items, obs, msk): import odc.stac import rioxarray as rxr import numpy as np import xarray as xr from shapely.geometry import box import pyproj # ----------------------------------------------------------- # 1) Metadata # ----------------------------------------------------------- crs_src = pyproj.CRS.from_wkt(items[0].properties["proj:wkt2"]) resolution = items[0].properties["gsd"] aoi = box(aa[1], aa[0], aa[3], aa[2]) # minx, miny, maxx, maxy sum_flood = None sum_missing = None print("\n=== Running flood extraction ===") # Loop one item at a time for idx, titem in enumerate(items): # ------------------------------------------------------- # 2) Load a single item (bbox + only 1 band) # ------------------------------------------------------- xx = odc.stac.load( [titem], bbox=aoi.bounds, crs=crs_src, bands=["ensemble_flood_extent","reference_water_mask"], resolution=resolution, dtype="uint8", groupby="solar_day", chunks={} # no dask ) da = xx["ensemble_flood_extent"] +xx["reference_water_mask"] #da = da + xx["reference_water_mask"] # ------------------------------------------------------- # 3) Coarsen BEFORE reproject (major speedup) # ------------------------------------------------------- factor = 4 da_ds = da.coarsen(y=factor, x=factor, boundary="trim").max() del da, xx # ------------------------------------------------------- # 4) Reproject # ------------------------------------------------------- da_ds = da_ds.rio.write_crs(crs_src) da_latlon = da_ds.rio.reproject( "EPSG:4326", nodata=255, resampling="average" #resampling="nearest" ) del da_ds # ------------------------------------------------------- # 5) Initialize accumulators using *raw numpy arrays* # → avoids xarray alignment problems # ------------------------------------------------------- if sum_flood is None: shape = da_latlon.shape sum_flood = np.zeros(shape, dtype="float32") sum_missing = np.zeros(shape, dtype="uint16") # ------------------------------------------------------- # 6) Convert input to numpy array once # ------------------------------------------------------- arr = da_latlon.fillna(255).astype("uint16").values del da_latlon # ------------------------------------------------------- # 7) Local computation on pure numpy (fastest) # ------------------------------------------------------- flood = np.where((arr != 255) & (arr != 0), arr, 0).astype("float32") missing = np.where((arr != 255),1,0).astype("uint16") del arr # ------------------------------------------------------- # 8) Accumulate (SAFE, no xarray alignment issues) # ------------------------------------------------------- sum_flood += flood sum_missing += missing del flood, missing print(f"Item {idx+1}/{len(items)} OK") # ----------------------------------------------------------- # 9) Convert numpy → DataArray for export # ----------------------------------------------------------- da_template = xr.zeros_like(items[0]["geometry"]) if False else None # just reuse the last projection from above last = odc.stac.load( [items[-1]], bbox=aoi.bounds, crs=crs_src, bands=["ensemble_flood_extent"], resolution=resolution, dtype="uint8", groupby="solar_day", chunks={} )["ensemble_flood_extent"].coarsen( y=factor, x=factor, boundary="trim" ).max().rio.write_crs(crs_src).rio.reproject("EPSG:4326") sum_flood_da = xr.DataArray(sum_flood,coords=last.coords, dims=last.dims).rio.write_crs("EPSG:4326") del last # ----------------------------------------------------------- # 10) Cleanup & Write Flood # ----------------------------------------------------------- sum_flood_da = sum_flood_da.fillna(0) sum_flood_da.values[np.isnan(sum_flood_da.values)] = 0 sum_flood_da.rio.to_raster(obs, compress="LZW") # ----------------------------------------------------------- # 11) Missing mask # ----------------------------------------------------------- msk_da = xr.DataArray( (sum_missing >= 1).astype("uint8"), coords=sum_flood_da.coords, dims=sum_flood_da.dims ).rio.write_crs("EPSG:4326") msk_da.values[np.isnan(msk_da.values)] = 0 msk_da.rio.to_raster(msk, compress="LZW") print("\n=== DONE ===\n") return crs_src

In [ ]:
# Define the area of interest (AOI), the date and days of the search

flood_case="SriLanka"
date = 20251126 ; days = 1
area = [5, 80, 10, 82] # SriLanka 

flood_case = "Thessaly"
date = 20230907 ; days = 1
area = [36, 20, 41, 25] # Thessaly

flood_case = "Germany"
date = 20210715 ; days = 3
area = [44, 0., 54.,10.] # Germany

flood_case = "Valencia"
date = 20241030 ; days = 2
area = [37, -2.5, 41,1.5] # Valencia

flood_case="Oregon"
date = 20251209 ; days = 4
area=[42, -128,52,-118] 

flood_case="EmiliaRomagna"
date = 20230503 ; days = 2
area = [46, 9, 43, 14] #EmiliaRomagna

flood_case="Pakistan"
date = 20220830 ; days = 1
area = [22, 66, 31, 72] # Pakistan
area = [24, 67.5, 29, 70] # Pakistan small area

flood_case="Australia"
date = 20250325 ; days = 5
date = 20251231 ; days = 1
date = 20260103 ; days = 1
area = [-35, 135, -10, 155] # Australia
area = [-20, 142, -15, 150] # Australia
area = [-25, 135, -10, 155]  # Australia

flood_case="SouthAfrica"
date = 20260124
area = [-15, 20, -35, 40]
area = [-20, 30, -28, 38]

flood_case="Spain"
date = 20260206
area = [31.5, -10, 40, 0] #Spain

flood_case="France"
date = 20260212 ; days = 2
area = [42, -2, 46, 2] #France

flood_case="Persia"
area = [35, 40, 30, 45]
area = [35, 43, 29, 50]
date = 20260330
days = 2

flood_case="SouthItaly"
area = [43, 12, 36, 19] #SouthItaly
date = 20260402
days = 2

flood_case="Yangtze"
area = [28, 105, 38, 125]  # Yangtze 
date = 20200722
days = 1

obs="/perm/pad/flood_cases/"+flood_case+"_flood_obs_GFM.tif"
msk="/perm/pad/flood_cases/"+flood_case+"_flood_mask_GFM.tif"

In [ ]:
# Search and extract. NOTE: Below an alternative version seful for large domains
items=search_subdomain(area,date,days)
print(f"On EODC we found {len(items)} items for the given search query")
if (len(items) > 0):
    trs=extract_subdomain(area,items,obs,msk)
    print(f"Coordinates {trs} ")

## Plot the EO area

Both the observation and the mask are plotted reducing the plot by a factor (10 typically)

In [ ]:
import rasterio
import matplotlib.pyplot as plt
from rasterio.enums import Resampling

path_obs  = "/perm/pad/flood_cases/"+flood_case+"_flood_obs_GFM.tif"
path_mask = "/perm/pad/flood_cases/"+flood_case+"_flood_mask_GFM.tif"

def read_downsampled(path, factor=10):
    with rasterio.open(path) as ds:
        arr = ds.read(
            1,
            out_shape=(
                1,
                ds.height // factor,
                ds.width // factor
            ),
            resampling=Resampling.nearest
        )
    return arr

obs  = read_downsampled(path_obs, factor=10)
mask = read_downsampled(path_mask, factor=10)

# Plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.imshow(obs, cmap="Blues",vmin=0,vmax=1)
plt.title(flood_case+" — Flood Observations")
plt.colorbar(fraction=0.046, pad=0.04)

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="Reds", vmin=0,vmax=1)
plt.title(flood_case+" — Flood Mask")
plt.colorbar(fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import rasterio
from rasterio.enums import Resampling

def read_downsampled(path, factor=10):
    with rasterio.open(path) as ds:
        data = ds.read(
            1,
            out_shape=(
                ds.height // factor,
                ds.width // factor
            ),
            resampling=Resampling.nearest
        )

        # Scale transform accordingly
        transform = ds.transform * ds.transform.scale(
            ds.width / data.shape[1],
            ds.height / data.shape[0]
        )

    return data, transform

def transform_to_extent(transform, shape):
    height, width = shape
    lon_min = transform.c
    lon_max = transform.c + transform.a * width
    lat_max = transform.f
    lat_min = transform.f + transform.e * height  # e < 0
    return [lon_min, lon_max, lat_min, lat_max]

flood_case="Pakistan"
date = 20220830 ; days = 1
area = [24, 67.5, 29, 70] # Pakistan
path_obs  = "/perm/pad/flood_cases/"+flood_case+"_flood_obs.tif"
path_mask = "/perm/pad/flood_cases/"+flood_case+"_flood_mask.tif"

obs,  obs_tr  = read_downsampled(path_obs,  factor=10)
mask, mask_tr = read_downsampled(path_mask, factor=10)
extent = transform_to_extent(obs_tr, obs.shape)

# --------------------------------------------------
# Plot geographic extent
# --------------------------------------------------
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.imshow(
    obs,
    cmap="Blues",
    vmin=0, vmax=1,
    extent=extent,
    origin="upper"   # 🔴 REQUIRED
)
plt.title(flood_case + " — Flood Observations")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.colorbar(fraction=0.046, pad=0.04)

plt.subplot(1, 2, 2)
plt.imshow(
    mask,
    cmap="Reds",
    vmin=0, vmax=1,
    extent=extent,
    origin="upper"
)
plt.title(flood_case + " — Flood Mask")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.colorbar(fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Tiling the domain area (useful for floods covering large domains)

The code below parcel the area domain in tiles of delta degrees that are mosaic at the end. This is useful to produce large domains.

In [ ]:
def make_subdomains(area, delta=1):
    """
    Split a geographic area into tiles of size delta × delta degrees.
    area = [south, west, north, east] ; delta = tile size in degrees
    """
    border=0.
    south, west, north, east = area
    subdomains = []
    lat = south-border
    while lat < north+border:
        next_lat = min(lat + delta, north)
        lon = west - border
        while lon < east+border:
            next_lon = min(lon + delta, east)
            subdomains.append([lat, lon, next_lat, next_lon])
            lon += delta
        lat += delta
    return subdomains

In [ ]:
#area = [22, 66, 31, 72] # Pakistan Large Area
tiles_deg = make_subdomains(area, delta=4.)
print ("Splitting into "+str(len(tiles_deg))+" domains ")
print (tiles_deg)

In [ ]:
#Removing previous tiles if present
!ls /perm/pad/flood_cases/{flood_case}_flood_obs_*.tif
!ls /perm/pad/flood_cases/{flood_case}_flood_mask_*.tif
!rm -rf /perm/pad/flood_cases/{flood_case}_flood_obs_*.tif
!rm -rf /perm/pad/flood_cases/{flood_case}_flood_mask_*.tif

In [ ]:
n=0
for tile in tiles_deg:
    items=[]
    items=search_subdomain(tile,date,days)
    print(f"{str(n+1)} / {str(len(tiles_deg))} On EODC found {len(items)} EO items for area={tile} ")
    obs="/perm/pad/flood_cases/"+flood_case+"_flood_obs_"+str(n)+".tif"
    msk="/perm/pad/flood_cases/"+flood_case+"_flood_mask_"+str(n)+".tif"
    if (len(items) > 0):
        xx=extract_subdomain(tile,items,obs,msk)
    n += 1

In [ ]:
#BBOX check for debugging difficult
debug=0
if (debug==1):
    import glob
    import rasterio
    from rasterio.merge import merge
    for f in sorted(glob.glob("/perm/pad/flood_cases/"+flood_case+"_flood_obs_*.tif")):
        with rasterio.open(f) as src:
            print(f)
            print(" CRS:", src.crs)
            print(" Transform:", src.transform)
            print(" Shape:", src.height, src.width)
            print(" Bounds:", src.bounds)
            print()

    for f in sorted(glob.glob("/perm/pad/flood_cases/"+flood_case+"_flood_obs.tif")):
        with rasterio.open(f) as src:
            print(f)
            print(" CRS:", src.crs)
            print(" Transform:", src.transform)
            print(" Shape:", src.height, src.width)
            print(" Bounds:", src.bounds)
            print()
            

## Merging tiles

The extracted tiles are remerged into a single observation and mask files.

In [ ]:
import rasterio
import numpy as np
import glob
from rasterio.windows import Window
from rasterio.transform import from_bounds

tiles = sorted(glob.glob("/perm/pad/flood_cases/"+flood_case+"_flood_obs_*.tif"))

# --- compute full geographic extent ---
minx = +1e9
miny = +1e9
maxx = -1e9
maxy = -1e9

for t in tiles:
    with rasterio.open(t) as src:
        b = src.bounds
        minx = min(minx, b.left)
        miny = min(miny, b.bottom)
        maxx = max(maxx, b.right)
        maxy = max(maxy, b.top)
        res = src.res  # assuming same for all tiles
        crs = src.crs

resx, resy = res
width  = int(np.round((maxx - minx) / resx))
height = int(np.round((maxy - miny) / abs(resy)))

transform = from_bounds(minx, miny, maxx, maxy, width, height)
out_nodata = 0.


In [ ]:
out_file = "/perm/pad/flood_cases/"+flood_case+"_flood_obs.tif"

profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": crs,
    "transform": transform,
    "nodata": out_nodata,
    "compress": "LZW",
}

with rasterio.open(out_file, "w", **profile) as dst:
    dst.write(
        np.full((1, height, width), out_nodata, dtype="float32")
    )


In [ ]:
def intersection(dst, src):
    """Compute intersection window in both dst and src pixel grids."""

    # Bounds in geographic coords
    L = max(dst.bounds.left,   src.bounds.left)
    R = min(dst.bounds.right,  src.bounds.right)
    B = max(dst.bounds.bottom, src.bounds.bottom)
    T = min(dst.bounds.top,    src.bounds.top)

    if L >= R or B >= T:
        return None  # no overlap

    # Convert to pixel coords in dst
    dst_win = rasterio.windows.from_bounds(L, B, R, T, dst.transform)
    dst_win = dst_win.round_offsets().round_lengths()

    # Convert to pixel coords in src
    src_win = rasterio.windows.from_bounds(L, B, R, T, src.transform)
    src_win = src_win.round_offsets().round_lengths()

    # Ensure sizes match exactly
    h = min(int(dst_win.height), int(src_win.height))
    w = min(int(dst_win.width),  int(src_win.width))

    dst_win = Window(dst_win.col_off, dst_win.row_off, w, h)
    src_win = Window(src_win.col_off, src_win.row_off, w, h)

    return dst_win, src_win


from rasterio.windows import Window

with rasterio.open(out_file, "r+") as dst:
    for t in tiles:
        print ("Processing tile: ",t)

        with rasterio.open(t) as src:
            res = intersection(dst, src)
            if res is None:
                continue
            dst_win, src_win = res

            # read blocks
            tile = src.read(1, window=src_win).astype("float32")
            existing = dst.read(1, window=dst_win).astype("float32")

            # handle nodata on tile
            nod = src.nodata
            if nod is not None:
                tile = np.where(tile == nod, np.nan, tile)
            else:
                tile = np.where(tile == 255.0, np.nan, tile)

            existing = np.where(existing == out_nodata, np.nan, existing)

            # merge using max (nan-safe)
            merged = np.fmax(existing, tile)
            merged = np.where(np.isnan(merged), out_nodata, merged)

            # write the block
            dst.write(merged, 1, window=dst_win)

print ("Done!",out_file)


In [ ]:
tiles = sorted(glob.glob("/perm/pad/flood_cases/"+flood_case+"_flood_mask_*.tif"))

# --- compute full geographic extent ---
minx = +1e9
miny = +1e9
maxx = -1e9
maxy = -1e9

for t in tiles:
    with rasterio.open(t) as src:
        b = src.bounds
        minx = min(minx, b.left)
        miny = min(miny, b.bottom)
        maxx = max(maxx, b.right)
        maxy = max(maxy, b.top)
        res = src.res  # assuming same for all tiles
        crs = src.crs

resx, resy = res
width  = int(np.round((maxx - minx) / resx))
height = int(np.round((maxy - miny) / abs(resy)))

transform = from_bounds(minx, miny, maxx, maxy, width, height)
out_nodata = 0.

out_file = "/perm/pad/flood_cases/"+flood_case+"_flood_mask.tif"

profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 1,
    "dtype": "float32",
    "crs": crs,
    "transform": transform,
    "nodata": out_nodata,
    "compress": "LZW",
}

with rasterio.open(out_file, "w", **profile) as dst:
    dst.write(
        np.full((1, height, width), out_nodata, dtype="float32")
    )

with rasterio.open(out_file, "r+") as dst:
    for t in tiles:
        print ("Processing tile: ",t)

        with rasterio.open(t) as src:
            res = intersection(dst, src)
            if res is None:
                continue
            dst_win, src_win = res

            # read blocks
            tile = src.read(1, window=src_win).astype("float32")
            existing = dst.read(1, window=dst_win).astype("float32")

            # merge using max (nan-safe)
            merged = np.fmax(existing, tile)
            merged = np.where(np.isnan(merged), out_nodata, merged)

            # write the block
            dst.write(merged, 1, window=dst_win)

print ("Done!",out_file)

In [ ]:
import rasterio
import matplotlib.pyplot as plt
from rasterio.enums import Resampling

path_obs  = "/perm/pad/flood_cases/"+flood_case+"_flood_obs.tif"
path_mask = "/perm/pad/flood_cases/"+flood_case+"_flood_mask.tif"

def read_downsampled(path, factor=10):
    with rasterio.open(path) as ds:
        arr = ds.read(
            1,
            out_shape=(
                1,
                ds.height // factor,
                ds.width // factor
            ),
            resampling=Resampling.nearest
        )
    return arr

obs  = read_downsampled(path_obs, factor=10)
mask = read_downsampled(path_mask, factor=10)

# Plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.imshow(obs, cmap="Blues")
plt.title(flood_case+" — Flood Observations")
plt.colorbar(fraction=0.046, pad=0.04)

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="Reds")
plt.title(flood_case+" — Flood Mask")
plt.colorbar(fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
